In [ ]:
from glob import glob
import os
from PIL import Image, ImageDraw
from tqdm import tqdm

from lxml import etree
import pandas as pd

### 2025-10-31_aero_3suture_sped


In [ ]:
DATA_DIR = "/home/chungk1/Repositories/ALISS-ARPAH/Anatomical-Landmark-Detector-Training/data/"
VIDEO_NAME = "2025-10-31_aero_3suture_sped"
DIR = f"{DATA_DIR}/raw/{VIDEO_NAME}/"
XML_PATH = os.path.join(DIR, "annotations.xml")
LEFT = 485

annotations = dict()

image_paths = glob(os.path.join(DIR, "images/*.PNG"))
for path in image_paths:
    filename = os.path.basename(path)
    frame = filename.split(".")[0].split("_")[1]
    annotations[int(frame)] = {"frame": int(frame), "video_name": VIDEO_NAME, "frame_name": filename}

root = etree.parse(XML_PATH).getroot()

for track in root.findall("track"):
    label = track.get("label")
    if (label != "needle_tip") and (label != "needle_tail"): 
        continue
    # print(f"Track Label: {label}")
    for points in track.findall("points"):
        frame = int(points.get("frame"))
        points = points.get("points")
        points = [float(val) for val in points.split(",")]

        annotations[frame][label] = points

annotations = [val for val in annotations.values()]
annotations.sort(key=lambda x: x["frame"])


for annotation in tqdm(annotations):
    image_path = os.path.join(DIR, "images", annotation["frame_name"])
    image = Image.open(image_path)
    crop = image.crop((LEFT, 0, LEFT + 1080, 1080))
    annotation["original_image"] = image_path
    if "needle_tip" in annotation.keys():
        annotation["landmark_1_x"] = annotation["needle_tip"][0] - LEFT
        annotation["landmark_1_y"] = annotation["needle_tip"][1]
    else:
        annotation["landmark_1_x"] = 0.0
        annotation["landmark_1_y"] = 0.0
    if "needle_tail" in annotation.keys():
        annotation["landmark_2_x"] = annotation["needle_tail"][0] - LEFT
        annotation["landmark_2_y"] = annotation["needle_tail"][1]
    else:
        annotation["landmark_2_x"] = 0.0
        annotation["landmark_2_y"] = 0.0
    image_name = annotation["video_name"] + "_" + annotation["frame_name"]
    annotation["image_name"] = image_name
    crop.save(os.path.join(DATA_DIR, "train_images", image_name))

df = pd.DataFrame(annotations)
df["image_width"] = 1080
df["image_height"] = 1080
df["n_landmarks"] = 2
df = df[["image_name", "image_width", "image_height", "n_landmarks", "landmark_1_x", "landmark_1_y", "landmark_2_x", "landmark_2_y"]]
df["landmark_1_x"] = df["landmark_1_x"].round(2)
df["landmark_1_y"] = df["landmark_1_y"].round(2)
df["landmark_2_x"] = df["landmark_2_x"].round(2)
df["landmark_2_y"] = df["landmark_2_y"].round(2)

df.to_csv(os.path.join(DIR, "train_annotations.csv"), index=False)

In [ ]:
# # Visualize the annotations
# R = 6
# for row in df.itertuples():
#     image = Image.open(os.path.join(DATA_DIR, "train_images", row.image_name))
#     draw = ImageDraw.Draw(image)
#     draw.ellipse((row.landmark_1_x - R, row.landmark_1_y - R, row.landmark_1_x + R, row.landmark_1_y + R), fill="red")
#     draw.ellipse((row.landmark_2_x - R, row.landmark_2_y - R, row.landmark_2_x + R, row.landmark_2_y + R), fill="blue")
#     image
#     break
# image

### 2026-02-24_grasping_trachea1

In [ ]:
DATA_DIR = "/home/chungk1/Repositories/ALISS-ARPAH/Anatomical-Landmark-Detector-Training/data"
# VIDEO_NAME = "2026-02-24_grasping_trachea1"
# VIDEO_NAME = "2026-02-24_trachea2"
VIDEO_NAME = "2026-02-24_trachea3"
DIR = f"{DATA_DIR}/raw/{VIDEO_NAME}/"
XML_PATH = os.path.join(DIR, "annotations.xml")
LEFT = 0

annotations = dict()

image_paths = glob(os.path.join(DIR, "images/*.png"))
for path in image_paths:
    filename = os.path.basename(path)
    frame = filename.split(".")[0].split("_")[1]
    annotations[int(frame)] = {"frame": int(frame), "video_name": VIDEO_NAME, "frame_name": filename}

root = etree.parse(XML_PATH).getroot()

for image in root.findall("image"):
    filename = image.get("name")
    frame = int(filename.split(".")[0].split("_")[1])
    for points in image.findall("points"):
        label = points.get("label")
        if (label != "needle_tip") and (label != "needle_tail"):
           continue
        
        points = points.get("points")
        points = [float(val) for val in points.split(",")]

        annotations[frame][label] = points

annotations = [val for val in annotations.values()]
annotations.sort(key=lambda x: x["frame"])

for annotation in tqdm(annotations):
    image_path = os.path.join(DIR, "images", annotation["frame_name"])
    image = Image.open(image_path)
    crop = image.crop((LEFT, 0, LEFT + 1080, 1080))
    annotation["original_image"] = image_path
    if "needle_tip" in annotation.keys():
        annotation["landmark_1_x"] = annotation["needle_tip"][0] - LEFT
        annotation["landmark_1_y"] = annotation["needle_tip"][1]
    else:
        annotation["landmark_1_x"] = 0.0
        annotation["landmark_1_y"] = 0.0
    if "needle_tail" in annotation.keys():
        annotation["landmark_2_x"] = annotation["needle_tail"][0] - LEFT
        annotation["landmark_2_y"] = annotation["needle_tail"][1]
    else:
        annotation["landmark_2_x"] = 0.0
        annotation["landmark_2_y"] = 0.0
    image_name = annotation["video_name"] + "_" + annotation["frame_name"]
    annotation["image_name"] = image_name
    crop.save(os.path.join(DATA_DIR, "train_images", image_name))

df = pd.DataFrame(annotations)
df["image_width"] = 1080
df["image_height"] = 1080
df["n_landmarks"] = 2
df = df[["image_name", "image_width", "image_height", "n_landmarks", "landmark_1_x", "landmark_1_y", "landmark_2_x", "landmark_2_y"]]
df["landmark_1_x"] = df["landmark_1_x"].round(2)
df["landmark_1_y"] = df["landmark_1_y"].round(2)
df["landmark_2_x"] = df["landmark_2_x"].round(2)
df["landmark_2_y"] = df["landmark_2_y"].round(2)

df.to_csv(os.path.join(DIR, "train_annotations.csv"), index=False)

In [ ]:
# Visualize the annotations
R = 6
for row in df.itertuples():
    image = Image.open(os.path.join(DATA_DIR, "train_images", row.image_name))
    draw = ImageDraw.Draw(image)
    draw.ellipse((row.landmark_1_x - R, row.landmark_1_y - R, row.landmark_1_x + R, row.landmark_1_y + R), fill="red")
    draw.ellipse((row.landmark_2_x - R, row.landmark_2_y - R, row.landmark_2_x + R, row.landmark_2_y + R), fill="blue")
    image
    break
image

### 2026-03-09_cadaver1

In [ ]:
DATA_DIR = "/home/chungk1/Repositories/ALISS-ARPAH/Anatomical-Landmark-Detector-Training/data"
VIDEO_NAME = "2026-03-09_cadaver1"
DIR = f"{DATA_DIR}/raw/{VIDEO_NAME}/"
XML_PATH = os.path.join(DIR, "annotations.xml")
LEFT = 470

annotations = dict()

image_paths = glob(os.path.join(DIR, "images/*.PNG"))
for path in image_paths:
    filename = os.path.basename(path)
    frame = filename.split(".")[0].split("_")[1]
    annotations[int(frame)] = {"frame": int(frame), "video_name": VIDEO_NAME, "frame_name": filename}

root = etree.parse(XML_PATH).getroot()

for track in root.findall("track"):
    label = track.get("label")
    if (label != "needle_tip") and (label != "needle_tail"): 
        continue
    # print(f"Track Label: {label}")
    for points in track.findall("points"):
        frame = int(points.get("frame"))
        points = points.get("points")
        points = [float(val) for val in points.split(",")]

        annotations[frame][label] = points

annotations = [val for val in annotations.values()]
annotations.sort(key=lambda x: x["frame"])

for annotation in tqdm(annotations):
    image_path = os.path.join(DIR, "images", annotation["frame_name"])
    image = Image.open(image_path)
    crop = image.crop((LEFT, 0, LEFT + 1080, 1080))
    annotation["original_image"] = image_path
    if "needle_tip" in annotation.keys():
        annotation["landmark_1_x"] = annotation["needle_tip"][0] - LEFT
        annotation["landmark_1_y"] = annotation["needle_tip"][1]
    else:
        annotation["landmark_1_x"] = 0.0
        annotation["landmark_1_y"] = 0.0
    if "needle_tail" in annotation.keys():
        annotation["landmark_2_x"] = annotation["needle_tail"][0] - LEFT
        annotation["landmark_2_y"] = annotation["needle_tail"][1]
    else:
        annotation["landmark_2_x"] = 0.0
        annotation["landmark_2_y"] = 0.0
    image_name = annotation["video_name"] + "_" + annotation["frame_name"]
    annotation["image_name"] = image_name
    crop.save(os.path.join(DATA_DIR, "train_images", image_name))

df = pd.DataFrame(annotations)
df["image_width"] = 1080
df["image_height"] = 1080
df["n_landmarks"] = 2
df = df[["image_name", "image_width", "image_height", "n_landmarks", "landmark_1_x", "landmark_1_y", "landmark_2_x", "landmark_2_y"]]
df["landmark_1_x"] = df["landmark_1_x"].round(2)
df["landmark_1_y"] = df["landmark_1_y"].round(2)
df["landmark_2_x"] = df["landmark_2_x"].round(2)
df["landmark_2_y"] = df["landmark_2_y"].round(2)

df.to_csv(os.path.join(DIR, "train_annotations.csv"), index=False)

In [ ]:
# Visualize the annotations
R = 6
for row in df.itertuples():
    image = Image.open(os.path.join(DATA_DIR, "train_images", row.image_name))
    draw = ImageDraw.Draw(image)
    draw.ellipse((row.landmark_1_x - R, row.landmark_1_y - R, row.landmark_1_x + R, row.landmark_1_y + R), fill="red")
    draw.ellipse((row.landmark_2_x - R, row.landmark_2_y - R, row.landmark_2_x + R, row.landmark_2_y + R), fill="blue")
    image
    break
image

### Combine all files

In [ ]:
csv_files = glob(os.path.join(DATA_DIR, "raw/*/train_annotations.csv"))
csv_files = [pd.read_csv(f) for f in csv_files]
df = pd.concat(csv_files, ignore_index=True)
df["landmark_1_x"] = df["landmark_1_x"].round(0).astype(int)
df["landmark_1_y"] = df["landmark_1_y"].round(0).astype(int)
df["landmark_2_x"] = df["landmark_2_x"].round(0).astype(int)
df["landmark_2_y"] = df["landmark_2_y"].round(0).astype(int)
df.to_csv(os.path.join(DATA_DIR, "labels", "train_annotation.csv"), index=False)